# ML-06 — Signal Audit: Do the Flags Hold?

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

Rule: Prioritize pages that haven't been updated for a long time and have a low CTR. These pages are likely candidates for a content refresh.

In [6]:
!git clone https://github.com/aqsaafzal483-cyber/flyrank-ml-internship2.git

Cloning into 'flyrank-ml-internship2'...
remote: Enumerating objects: 131, done.
remote: Counting objects: 100% (131/131), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 131 (delta 44), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (131/131), 1.85 MiB | 10.71 MiB/s, done.
Resolving deltas: 100% (44/44), done.


In [7]:
%cd flyrank-ml-internship2

/content/flyrank-ml-internship2


Verdict: MIXED

What this means in practice:
The data does not strongly confirm that older content always needs refreshing.
Freshness should be used as a supporting signal because the oldest buckets have very few examples.

Verdict: CONFIRMED

What this means in practice:
Low CTR pages appear to be good candidates for improvement.
CTR can be used as a useful signal for identifying pages that may need search appearance optimization.

In [8]:
!ls

01_first_look_and_discovery.ipynb   DATA_USE.md  outputs	   skills
02_your_first_readable_model.ipynb  docs	 README.md	   submission
AGENTS.md			    GUIDE.md	 requirements.txt  work
CLAUDE.md			    LICENSE	 scripts
data				    notebooks	 SETUP.md


In [9]:


import pandas as pd

In [10]:
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

In [11]:
df.shape

(30000, 44)

In [12]:
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [13]:
df.columns

Index(['content_id', 'client_id', 'search_volume', 'competition',
       'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count',
       'char_count', 'provider_used', 'model_used', 'impressions_90d',
       'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
       'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
       'days_with_impressions', 'days_with_sessions', 'impressions_last_30d',
       'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d',
       'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier',
       'age_tier_order', 'days_since_last_update', 'freshness_tier',
       'word_count_tier', 'char_count_tier', 'ctr', 'avg_position',
       'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier',
       'position_tier', 'trend_direction', 'trend_pct'],
      dtype='object')

In [19]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'update_bucket', 'ctr_bucket']


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [15]:
df["days_since_last_update"].describe()

,days_since_last_update
count,30000.000000
mean,46.098300
std,42.078709
min,1.000000
25%,20.000000
50%,20.000000
75%,104.000000
max,373.000000


In [16]:
df["update_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[0, 90, 180, 365, 10000],
    labels=["0-90", "91-180", "181-365", "365+"]
)

bucket1 = df.groupby("update_bucket").agg(
    n=("content_id", "count"),
    avg_ctr=("ctr", "mean")
)

bucket1

/tmp/ipykernel_1498/3107142743.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket1 = df.groupby("update_bucket").agg(


,n,avg_ctr
update_bucket,,
0-90,20655,0.604856
91-180,9171,0.238367
181-365,169,3.210828
365+,5,20.000000


In [17]:
df["ctr_bucket"] = pd.cut(
    df["ctr"],
    bins=[0, 1, 2, 5, 100],
    labels=["0-1%", "1-2%", "2-5%", "5%+"]
)

bucket2 = df.groupby("ctr_bucket").agg(
    n=("content_id", "count"),
    avg_position=("avg_position", "mean")
)

bucket2

/tmp/ipykernel_1498/797823259.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket2 = df.groupby("ctr_bucket").agg(


,n,avg_position
ctr_bucket,,
0-1%,15099,14.663342
1-2%,915,11.658142
2-5%,333,10.507508
5%+,441,7.869615


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [20]:
import numpy as np

df["baseline_score"] = 0

df.loc[df["ctr_bucket"] == "0-1%", "baseline_score"] += 2
df.loc[df["ctr_bucket"] == "1-2%", "baseline_score"] += 1

df.loc[df["update_bucket"].isin(["181-365", "365+"]), "baseline_score"] += 1

In [21]:
df["reason_code"] = "MONITOR"

df.loc[
    (df["ctr_bucket"] == "0-1%") &
    (df["update_bucket"].isin(["181-365", "365+"])),
    "reason_code"
] = "LOW_CTR_STALE"

df.loc[
    (df["ctr_bucket"] == "0-1%") &
    (~df["update_bucket"].isin(["181-365", "365+"])),
    "reason_code"
] = "LOW_CTR"

df.loc[
    (df["ctr_bucket"] != "0-1%") &
    (df["update_bucket"].isin(["181-365", "365+"])),
    "reason_code"
] = "STALE_CONTENT"

In [22]:
df["action_label"] = "Monitor"

df.loc[df["reason_code"] == "LOW_CTR_STALE", "action_label"] = "Refresh Content"
df.loc[df["reason_code"] == "LOW_CTR", "action_label"] = "Improve CTR"
df.loc[df["reason_code"] == "STALE_CONTENT", "action_label"] = "Refresh Content"

In [24]:
import os

os.makedirs("work/outputs", exist_ok=True)

queue = df.sort_values("baseline_score", ascending=False)

queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Saved successfully!")

Saved successfully!


In [25]:
import os
print(os.getcwd())

/content/flyrank-ml-internship2


In [26]:
!ls

01_first_look_and_discovery.ipynb   DATA_USE.md  outputs	   skills
02_your_first_readable_model.ipynb  docs	 README.md	   submission
AGENTS.md			    GUIDE.md	 requirements.txt  work
CLAUDE.md			    LICENSE	 scripts
data				    notebooks	 SETUP.md


In [27]:
import os
print(os.getcwd())

/content/flyrank-ml-internship2


In [28]:
import os
print(os.getcwd())
!ls

/content/flyrank-ml-internship2
01_first_look_and_discovery.ipynb   DATA_USE.md  outputs	   skills
02_your_first_readable_model.ipynb  docs	 README.md	   submission
AGENTS.md			    GUIDE.md	 requirements.txt  work
CLAUDE.md			    LICENSE	 scripts
data				    notebooks	 SETUP.md


In [29]:
import os

os.makedirs("work/outputs", exist_ok=True)

queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

print("CSV saved successfully!")

CSV saved successfully!


In [30]:
!ls work

capstone_report_template.md  notebooks	outputs  README.md


In [31]:
import os

os.makedirs("work/outputs", exist_ok=True)

In [32]:
queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Saved successfully!")

Saved successfully!


In [33]:
!pwd
!ls work
!ls work/outputs

/content/flyrank-ml-internship2
capstone_report_template.md  notebooks	outputs  README.md
baseline_action_score.csv


In [35]:
top10 = queue.head(10)

top10

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,update_bucket,ctr_bucket,baseline_score,reason_code,action_label
16514,content_7368877ea310,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,2591.0,16498.0,...,0.0,excellent,page_3_5,down,-81.5,181-365,0-1%,3,LOW_CTR_STALE,Refresh Content
7452,content_72496874f806,client_4ec9599fc2,NaN,NaN,NaN,NaN,keyword article,NaN,1504.0,10770.0,...,0.0,moderate,page_1,down,-22.4,181-365,0-1%,3,LOW_CTR_STALE,Refresh Content
26810,content_ecb6215e79fd,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,4486.0,29333.0,...,0.0,good,page_3_5,down,-74.4,181-365,0-1%,3,LOW_CTR_STALE,Refresh Content
26799,content_77d4d5930e5e,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,4020.0,26513.0,...,0.0,moderate,striking,down,-55.0,181-365,0-1%,3,LOW_CTR_STALE,Refresh Content
5327,content_fe16a55cd13d,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,3388.0,21742.0,...,0.0,good,striking,down,-52.2,181-365,0-1%,3,LOW_CTR_STALE,Refresh Content
23741,content_df1fa766cac2,client_9400f1b21c,NaN,NaN,NaN,NaN,keyword article,NaN,1146.0,8277.0,...,0.0,low,page_1,down,-97.7,181-365,0-1%,3,LOW_CTR_STALE,Refresh Content
11489,content_5feee3994adb,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,transactional,3590.0,22780.0,...,0.0,good,page_3_5,down,-89.1,181-365,0-1%,3,LOW_CTR_STALE,Refresh Content
16751,content_cf56e2e2e282,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,5125.0,33705.0,...,0.0,excellent,striking,down,-85.6,181-365,0-1%,3,LOW_CTR_STALE,Refresh Content
26840,content_7f116ae1f6f5,client_9400f1b21c,NaN,NaN,NaN,NaN,keyword article,NaN,1335.0,9375.0,...,0.0,moderate,page_1,down,-44.5,181-365,0-1%,3,LOW_CTR_STALE,Refresh Content
23215,content_bdbec75c1148,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,3696.0,24643.0,...,0.0,moderate,page_3_5,stable,-11.7,181-365,0-1%,3,LOW_CTR_STALE,Refresh Content


### Top 10 Manual Reviews

The following pages were selected by the baseline scoring rule.
Each review explains why the page was prioritized and what could make the recommendation incorrect.

Review 1

Content ID: content_7368877ea310

Action: Refresh Content

Why it was selected:
This page has very low CTR (0-1%) and stale content (181-365 days), giving it the highest baseline priority score.

What could make it wrong:
The low CTR may be caused by low search demand or ranking position rather than outdated content.


Review 2

Content ID: content_72496874f806

Action: Refresh Content

Why it was selected:
The page has low CTR and has not been updated for a long time, matching the LOW_CTR_STALE rule.

What could make it wrong:
The content may still be accurate and useful, so updating it may not improve performance.


Review 3

Content ID: content_ecb6215e79fd

Action: Refresh Content

Why it was selected:
This page was prioritized because it combines weak CTR performance with stale freshness signals.

What could make it wrong:
The issue may be related to search intent mismatch or competition instead of content freshness.


Review 4

Content ID: content_77d4d5930e5e

Action: Refresh Content

Why it was selected:
The page received a high score because it has a low CTR and has not been updated recently.

What could make it wrong:
A refresh may not help if users are already satisfied with the existing content.


Review 5

Content ID: content_fe16a55cd13d

Action: Refresh Content

Why it was selected:
The page shows both low CTR and older update history, making it a candidate for content review.

What could make it wrong:
The CTR may naturally be low for the targeted query type.


Review 6

Content ID: content_df1fa766cac2

Action: Refresh Content

Why it was selected:
The page has stale content signals and very low CTR, which suggests possible improvement opportunities.

What could make it wrong:
The traffic decline may come from changes in search behavior or seasonality.


Review 7

Content ID: content_5feee3994adb

Action: Refresh Content

Why it was selected:
This page matches the baseline rule because it has low CTR and older content freshness.

What could make it wrong:
The page may require SEO improvements such as targeting or ranking changes instead of a content refresh.


Review 8

Content ID: content_cf56e2e2e282

Action: Refresh Content

Why it was selected:
The page was ranked highly because it has both CTR weakness and stale content indicators.

What could make it wrong:
The page may still provide value despite having weak click performance.


Review 9

Content ID: content_7f116ae1f6f5

Action: Refresh Content

Why it was selected:
The baseline model selected this page due to the combination of low CTR and old update timing.

What could make it wrong:
Low CTR could be caused by poor ranking position rather than content quality.


Review 10

Content ID: content_bdbec75c1148

Action: Refresh Content

Why it was selected:
The page has the two main risk signals used by the baseline rule: low CTR and stale content.

What could make it wrong:
The correct action may be improving search visibility instead of refreshing the content.

In [49]:
queue = df.sort_values(
    "baseline_score",
    ascending=False
)

In [50]:
top10 = queue.head(10).copy()

In [51]:
top10.insert(0, "rank", range(1, 11))

In [52]:
top10[
    [
        "rank",
        "content_id",
        "baseline_score",
        "reason_code",
        "action_label"
    ]
]

,rank,content_id,baseline_score,reason_code,action_label
16514,1,content_7368877ea310,3,LOW_CTR_STALE,Refresh Content
7452,2,content_72496874f806,3,LOW_CTR_STALE,Refresh Content
26810,3,content_ecb6215e79fd,3,LOW_CTR_STALE,Refresh Content
26799,4,content_77d4d5930e5e,3,LOW_CTR_STALE,Refresh Content
5327,5,content_fe16a55cd13d,3,LOW_CTR_STALE,Refresh Content
23741,6,content_df1fa766cac2,3,LOW_CTR_STALE,Refresh Content
11489,7,content_5feee3994adb,3,LOW_CTR_STALE,Refresh Content
16751,8,content_cf56e2e2e282,3,LOW_CTR_STALE,Refresh Content
26840,9,content_7f116ae1f6f5,3,LOW_CTR_STALE,Refresh Content
23215,10,content_bdbec75c1148,3,LOW_CTR_STALE,Refresh Content


In [36]:
df.groupby("update_bucket")["ai_sessions_90d"].agg(["count","mean"])

/tmp/ipykernel_1498/1084952129.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("update_bucket")["ai_sessions_90d"].agg(["count","mean"])


,count,mean
update_bucket,,
0-90,20655,0.088260
91-180,9171,0.469305
181-365,169,0.047337
365+,5,0.000000


In [37]:
df.groupby("ctr_bucket")["ai_sessions_90d"].agg(["count","mean"])

/tmp/ipykernel_1498/3218813798.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("ctr_bucket")["ai_sessions_90d"].agg(["count","mean"])


,count,mean
ctr_bucket,,
0-1%,15099,0.334592
1-2%,915,0.439344
2-5%,333,0.162162
5%+,441,0.079365


In [38]:
import numpy as np

df["baseline_score"] = 0

# Low CTR pages get higher priority
df.loc[df["ctr_bucket"] == "0-1%", "baseline_score"] += 2
df.loc[df["ctr_bucket"] == "1-2%", "baseline_score"] += 1

# Older content gets priority
df.loc[df["update_bucket"].isin(["181-365", "365+"]), "baseline_score"] += 1

In [39]:
df["reason_code"] = "MONITOR"

df.loc[
    (df["ctr_bucket"] == "0-1%") &
    (df["update_bucket"].isin(["181-365", "365+"])),
    "reason_code"
] = "LOW_CTR_STALE"

In [40]:
df["action_label"] = "Monitor"

df.loc[
    df["reason_code"] == "LOW_CTR_STALE",
    "action_label"
] = "Refresh Content"

In [41]:
queue = df.sort_values(
    "baseline_score",
    ascending=False
)

queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

queue.head(10)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,update_bucket,ctr_bucket,baseline_score,reason_code,action_label
16514,content_7368877ea310,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,2591.0,16498.0,...,0.0,excellent,page_3_5,down,-81.5,181-365,0-1%,3,LOW_CTR_STALE,Refresh Content
7452,content_72496874f806,client_4ec9599fc2,NaN,NaN,NaN,NaN,keyword article,NaN,1504.0,10770.0,...,0.0,moderate,page_1,down,-22.4,181-365,0-1%,3,LOW_CTR_STALE,Refresh Content
26810,content_ecb6215e79fd,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,4486.0,29333.0,...,0.0,good,page_3_5,down,-74.4,181-365,0-1%,3,LOW_CTR_STALE,Refresh Content
26799,content_77d4d5930e5e,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,4020.0,26513.0,...,0.0,moderate,striking,down,-55.0,181-365,0-1%,3,LOW_CTR_STALE,Refresh Content
5327,content_fe16a55cd13d,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,3388.0,21742.0,...,0.0,good,striking,down,-52.2,181-365,0-1%,3,LOW_CTR_STALE,Refresh Content
23741,content_df1fa766cac2,client_9400f1b21c,NaN,NaN,NaN,NaN,keyword article,NaN,1146.0,8277.0,...,0.0,low,page_1,down,-97.7,181-365,0-1%,3,LOW_CTR_STALE,Refresh Content
11489,content_5feee3994adb,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,transactional,3590.0,22780.0,...,0.0,good,page_3_5,down,-89.1,181-365,0-1%,3,LOW_CTR_STALE,Refresh Content
16751,content_cf56e2e2e282,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,5125.0,33705.0,...,0.0,excellent,striking,down,-85.6,181-365,0-1%,3,LOW_CTR_STALE,Refresh Content
26840,content_7f116ae1f6f5,client_9400f1b21c,NaN,NaN,NaN,NaN,keyword article,NaN,1335.0,9375.0,...,0.0,moderate,page_1,down,-44.5,181-365,0-1%,3,LOW_CTR_STALE,Refresh Content
23215,content_bdbec75c1148,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,3696.0,24643.0,...,0.0,moderate,page_3_5,stable,-11.7,181-365,0-1%,3,LOW_CTR_STALE,Refresh Content


In [42]:
queue.head(10)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,update_bucket,ctr_bucket,baseline_score,reason_code,action_label
16514,content_7368877ea310,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,2591.0,16498.0,...,0.0,excellent,page_3_5,down,-81.5,181-365,0-1%,3,LOW_CTR_STALE,Refresh Content
7452,content_72496874f806,client_4ec9599fc2,NaN,NaN,NaN,NaN,keyword article,NaN,1504.0,10770.0,...,0.0,moderate,page_1,down,-22.4,181-365,0-1%,3,LOW_CTR_STALE,Refresh Content
26810,content_ecb6215e79fd,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,4486.0,29333.0,...,0.0,good,page_3_5,down,-74.4,181-365,0-1%,3,LOW_CTR_STALE,Refresh Content
26799,content_77d4d5930e5e,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,4020.0,26513.0,...,0.0,moderate,striking,down,-55.0,181-365,0-1%,3,LOW_CTR_STALE,Refresh Content
5327,content_fe16a55cd13d,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,3388.0,21742.0,...,0.0,good,striking,down,-52.2,181-365,0-1%,3,LOW_CTR_STALE,Refresh Content
23741,content_df1fa766cac2,client_9400f1b21c,NaN,NaN,NaN,NaN,keyword article,NaN,1146.0,8277.0,...,0.0,low,page_1,down,-97.7,181-365,0-1%,3,LOW_CTR_STALE,Refresh Content
11489,content_5feee3994adb,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,transactional,3590.0,22780.0,...,0.0,good,page_3_5,down,-89.1,181-365,0-1%,3,LOW_CTR_STALE,Refresh Content
16751,content_cf56e2e2e282,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,5125.0,33705.0,...,0.0,excellent,striking,down,-85.6,181-365,0-1%,3,LOW_CTR_STALE,Refresh Content
26840,content_7f116ae1f6f5,client_9400f1b21c,NaN,NaN,NaN,NaN,keyword article,NaN,1335.0,9375.0,...,0.0,moderate,page_1,down,-44.5,181-365,0-1%,3,LOW_CTR_STALE,Refresh Content
23215,content_bdbec75c1148,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,3696.0,24643.0,...,0.0,moderate,page_3_5,stable,-11.7,181-365,0-1%,3,LOW_CTR_STALE,Refresh Content


## Weak Picks

1. Some pages may be incorrectly prioritized because low CTR does not always mean the content is poor.
Users may be searching for information that does not require clicking.

2. Older content may not always need a refresh. A page can remain accurate and useful even if it has not been updated recently.

3. The rule uses only a few signals (CTR and freshness), so it may miss other important factors such as search volume, ranking position, or engagement.

4. Small buckets with fewer examples may create unreliable patterns, so some recommendations need manual review before taking action.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

Content freshness alone is not a strong enough signal to prioritize pages.
The older content buckets show higher AI sessions, but those buckets have very few rows,
so the result may be influenced by small sample size. I should use freshness as a supporting
signal rather than the main reason for action.

CTR is a useful signal for identifying pages that may need improvement.
Pages with lower CTR have higher AI session averages in this dataset, suggesting that
search appearance improvements such as title or snippet optimization may help increase clicks.

## Self-check

✅ Two signal checks completed with bucket tables and n values.

✅ At least one signal was linked to a real FlyRank flag.

✅ Verdicts were added (CONFIRMED/MIXED/etc.).

✅ One baseline rule was created with:
- score
- reason code
- action label

✅ Ranked queue was saved to:
work/outputs/baseline_action_score.csv

✅ Top-10 rows were reviewed with:
- action
- why it was selected
- what could make it wrong

✅ No future-window or label-derived inputs were used.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.